In [ ]:
# --- CELL 1 ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import drive

# --- ADDED IMPORTS ---
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, SpatialDropout2D, BatchNormalization, Activation

drive.mount('/content/drive')
print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# --- CELL 2 ---
IMAGE_DIR = '/content/drive/MyDrive/Oil_Spill_Project/data/train/images'
MASK_DIR = '/content/drive/MyDrive/Oil_Spill_Project/data/train/labels'
MODEL_SAVE_DIR = '/content/drive/MyDrive/Oil_Spill_Project/saved_models'
AIS_DATA_PATH = '/content/drive/MyDrive/Oil_Spill_Project/data/ais_data/vessel_data_clean.csv'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

FINAL_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'sparse_unet_oil_spill.keras')

IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3
BATCH_SIZE = 16
EPOCHS = 40

In [ ]:
# --- CELL 3 ---
def load_image_and_mask(image_path, mask_path):
    img = tf.io.read_file(image_path)
    # Automatically handles .jpg, .jpeg, or .png
    img = tf.io.decode_image(img, channels=IMG_CHANNELS, expand_animations=False)
    img.set_shape([None, None, IMG_CHANNELS])
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    mask = tf.io.read_file(mask_path)
    # Masks strictly stay .png to preserve 0 and 1 pixel values
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.convert_image_dtype(mask, tf.float32)
    mask = tf.image.resize(mask, [IMG_HEIGHT, IMG_WIDTH], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    mask = tf.where(mask > 0.5, 1.0, 0.0)

    return img, mask

def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)

    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    return image, mask

def get_dataset(image_paths, mask_paths, batch_size, augment_data=True):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=1000)

    if augment_data:
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# Dynamically find the images whether they are jpg or png
valid_img_exts = ('.jpg', '.jpeg', '.png')
all_image_paths = sorted([os.path.join(IMAGE_DIR, fname) for fname in os.listdir(IMAGE_DIR) if fname.lower().endswith(valid_img_exts)])
all_mask_paths = sorted([os.path.join(MASK_DIR, fname) for fname in os.listdir(MASK_DIR) if fname.lower().endswith('.png')])

if len(all_image_paths) == 0 or len(all_mask_paths) == 0:
    raise ValueError(f"Found {len(all_image_paths)} images and {len(all_mask_paths)} masks. Check your folders and file extensions!")
if len(all_image_paths) != len(all_mask_paths):
    print(f"WARNING: Mismatch in count! {len(all_image_paths)} images vs {len(all_mask_paths)} masks.")

# --- DATA SPLITTING (Train/Validation) ---
X_train, X_val, y_train, y_val = train_test_split(
    all_image_paths, all_mask_paths,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print(f"Training shapes: {len(X_train)}, {len(y_train)}")
print(f"Validation shapes: {len(X_val)}, {len(y_val)}")

# Convert to tf.data.Dataset for optimized pipeline
train_dataset = get_dataset(X_train, y_train, BATCH_SIZE, augment_data=True)

# Validation set mapping (no augmentation, no shuffling)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Train & Validation Datasets successfully loaded with real images!")

In [ ]:
# --- CELL 4 ---
from tensorflow.keras import backend as K

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def focal_loss(y_true, y_pred, alpha=0.75, gamma=2.0):
    y_true = tf.cast(y_true, tf.float32)
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)

    # Calculate cross entropy
    cross_entropy = -y_true * K.log(y_pred)

    # Calculate focal weight
    weight = alpha * K.pow(1.0 - y_pred, gamma)

    loss = weight * cross_entropy
    return K.mean(K.sum(loss, axis=-1))

def sparse_focal_dice_loss(y_true, y_pred):
    return 0.5 * focal_loss(y_true, y_pred) + 0.5 * dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred > 0.5, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [ ]:
# --- CELL 5 ---
def build_sparse_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    inputs = Input(input_shape)

    # Encoder
    c1 = Conv2D(32, (3, 3), padding='same')(inputs)
    c1 = BatchNormalization()(c1)
    c1 = Activation('relu')(c1)
    c1 = Conv2D(32, (3, 3), padding='same')(c1)
    c1 = BatchNormalization()(c1)
    c1 = Activation('relu')(c1)
    p1 = MaxPooling2D((2, 2))(c1)
    p1 = SpatialDropout2D(0.1)(p1) # Added SpatialDropout

    c2 = Conv2D(64, (3, 3), padding='same')(p1)
    c2 = BatchNormalization()(c2)
    c2 = Activation('relu')(c2)
    c2 = Conv2D(64, (3, 3), padding='same')(c2)
    c2 = BatchNormalization()(c2)
    c2 = Activation('relu')(c2)
    p2 = MaxPooling2D((2, 2))(c2)
    p2 = SpatialDropout2D(0.2)(p2)

    # Bottleneck
    c3 = Conv2D(128, (3, 3), padding='same')(p2)
    c3 = BatchNormalization()(c3)
    c3 = Activation('relu')(c3)
    c3 = Conv2D(128, (3, 3), padding='same')(c3)
    c3 = BatchNormalization()(c3)
    c3 = Activation('relu')(c3)

    # Decoder
    u4 = UpSampling2D((2, 2))(c3)
    u4 = concatenate([u4, c2])
    c4 = Conv2D(64, (3, 3), padding='same')(u4)
    c4 = BatchNormalization()(c4)
    c4 = Activation('relu')(c4)
    c4 = SpatialDropout2D(0.2)(c4)
    c4 = Conv2D(64, (3, 3), padding='same')(c4)
    c4 = BatchNormalization()(c4)
    c4 = Activation('relu')(c4)

    u5 = UpSampling2D((2, 2))(c4)
    u5 = concatenate([u5, c1])
    c5 = Conv2D(32, (3, 3), padding='same')(u5)
    c5 = BatchNormalization()(c5)
    c5 = Activation('relu')(c5)
    c5 = Conv2D(32, (3, 3), padding='same')(c5)
    c5 = BatchNormalization()(c5)
    c5 = Activation('relu')(c5)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

model = build_sparse_unet((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=sparse_focal_dice_loss,
              metrics=[iou_metric])

model.summary()

In [ ]:
# --- CELL 6 ---
callbacks = [
    ModelCheckpoint(FINAL_MODEL_PATH, monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# --- CELL 7 ---
plt.figure(figsize=(15, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Sparse Focal + Dice Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot IoU
plt.subplot(1, 2, 2)
plt.plot(history.history['iou_metric'], label='Train IoU')
plt.plot(history.history['val_iou_metric'], label='Val IoU')
plt.title('Intersection Over Union Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# --- CELL 8 ---
def load_and_filter_ais(ais_path, bounds, lat_col='LAT', lon_col='LON'):
    if not os.path.exists(ais_path):
        return None

    df = pd.read_csv(ais_path)
    min_lat, max_lat, min_lon, max_lon = bounds

    filtered_df = df[
        (df[lat_col] >= min_lat) & (df[lat_col] <= max_lat) &
        (df[lon_col] >= min_lon) & (df[lon_col] <= max_lon)
    ]
    return filtered_df

sar_image_bounds = (28.0, 29.0, -91.0, -90.0)
nearby_vessels = load_and_filter_ais(AIS_DATA_PATH, sar_image_bounds)

if nearby_vessels is not None and not nearby_vessels.empty:
    display(nearby_vessels.head())